# 00 — Dataset audit (Kermany pediatric CXR)

**Goal:** Table 1 of the paper. For the official Kaggle split, measure:
1. exact duplicate images (identical pixels), within and across splits, and label conflicts
2. near-duplicates (perceptual hash)
3. how many filename-derived patient groups appear in both train and test
4. whether a shared filename ID across train/test really looks like the same patient

**Setup:** attach dataset `paultimothymooney/chest-xray-pneumonia`, Internet **on**, accelerator **None** (CPU is enough).
Outputs go to `/kaggle/working/audit/`.

In [ ]:
!pip install -q --no-deps --force-reinstall --no-cache-dir git+https://github.com/meap11/ppneu.git

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ppneu import audit as A

OUT = Path("/kaggle/working/audit"); OUT.mkdir(parents=True, exist_ok=True)
root, candidates = A.find_root("/kaggle/input")
print("using:", root)
print("all candidate roots:", *candidates, sep="\n  ")

## 1. Build the index (hashes every image; ~2–5 min on CPU)

In [ ]:
df = A.build_index(root, workers=os.cpu_count())
df.to_csv(OUT / "index.csv", index=False)
print(len(df), "images")
display(pd.crosstab(df["split"], df["label"], margins=True))
display(pd.crosstab(df["split"], df["subtype"], margins=True))

In [ ]:
unparsed = df[df["subtype"] == "unknown"]
print("unparsed filenames:", len(unparsed))
display(unparsed[["relpath"]].head(20))
print("\nimage modes:\n", df["mode"].value_counts())
display(df.groupby("split")[["width", "height"]].describe().T)

## 2. Exact duplicates (identical pixels)

In [ ]:
dups = A.exact_duplicates(df, key="md5_pixels")
dups.to_csv(OUT / "exact_duplicates.csv", index=False)
n_extra = int((dups["n"] - 1).sum())
print(f"duplicate sets: {len(dups)} | redundant images: {n_extra} | unique images: {len(df) - n_extra}")
print("cross-split sets:", int(dups["cross_split"].sum()))
print("label-conflict sets:", int(dups["label_conflict"].sum()))
print("subtype-conflict sets (bacteria vs virus):", int(dups["subtype_conflict"].sum()))
print("images in byte-identical sets:", df.duplicated("md5_bytes", keep=False).sum(), "images")
display(dups.sort_values(["cross_split", "label_conflict"], ascending=False).head(20))

## 3. Near-duplicates (pHash Hamming distance ≤ 4 of 64 bits)

In [ ]:
near = A.near_duplicate_pairs(df, max_dist=4)
near.to_csv(OUT / "near_duplicate_pairs.csv", index=False)
near_only = near[~near["exact"]]
print("near-duplicate pairs (excluding exact):", len(near_only))
print("  cross-split:", int(near_only["cross_split"].sum()))
print("  label conflict:", int(near_only["label_conflict"].sum()))
print("  same filename group:", int(near_only["same_group"].sum()))
display(near_only.sort_values("dist").head(20)[
    ["filename_a", "split_a", "label_a", "filename_b", "split_b", "label_b", "dist"]])

In [ ]:
# Look at a few near-duplicate pairs by eye: are they really the same radiograph?
show = near_only.sort_values("dist").head(6)
if len(show):
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)))
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(show.iterrows()):
        for c, side in enumerate("ab"):
            rel = df.loc[df["image_id"] == row[f"image_id_{side}"], "relpath"].item()
            axes[r, c].imshow(plt.imread(root / rel), cmap="gray")
            axes[r, c].set_title(f"{row[f'split_{side}']}/{row[f'filename_{side}']}", fontsize=8)
            axes[r, c].axis("off")
    plt.tight_layout(); plt.savefig(OUT / "near_dup_examples.png", dpi=120); plt.show()

## 4. Patient-group overlap between splits

In [ ]:
overlap = {col: A.group_overlap(df, col) for col in ["group_fine", "group_coarse"]}
display(pd.DataFrame(overlap))

## 5. Do shared IDs across train/test mean the same patient?

If the numbering restarted in each folder, `person12` in train and `person12` in test could be different children.
We compare pHash distances of three kinds of pairs. Random 64-bit hashes differ by ~32 bits on average.

In [ ]:
ic = A.id_consistency(df, col="group_fine", n_random=20000)
summary_ic = ic.groupby("kind")["dist"].describe()
display(summary_ic)

fig, ax = plt.subplots(figsize=(7, 4))
for kind, g in ic.groupby("kind"):
    ax.hist(g["dist"], bins=range(0, 50), density=True, histtype="step", linewidth=2, label=f"{kind} (n={len(g)})")
ax.set_xlabel("pHash Hamming distance"); ax.set_ylabel("density"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "id_consistency.png", dpi=120); plt.show()

## 6. Save the summary (this becomes Table 1)

In [ ]:
summary = {
    "dataset_root": str(root),
    "n_images": int(len(df)),
    "counts": pd.crosstab(df["split"], df["label"]).to_dict(),
    "n_unparsed_names": int((df["subtype"] == "unknown").sum()),
    "exact_duplicate_sets": int(len(dups)),
    "exact_redundant_images": n_extra,
    "unique_images": int(len(df) - n_extra),
    "exact_cross_split_sets": int(dups["cross_split"].sum()),
    "exact_label_conflict_sets": int(dups["label_conflict"].sum()),
    "near_dup_pairs_non_exact": int(len(near_only)),
    "near_dup_cross_split": int(near_only["cross_split"].sum()),
    "near_dup_label_conflict": int(near_only["label_conflict"].sum()),
    "group_overlap": overlap,
    "id_consistency_median": ic.groupby("kind")["dist"].median().to_dict(),
}
(OUT / "audit_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))